# Download US Macro-Event Release Dates → Daily Indicator

Companion to `Download_price_Data.ipynb`. This notebook does **not** re-download any price data.
Instead it collects the US release / decision dates for four high-impact macro events and turns
them into a daily **indicator function** that can be merged into the HAR datasets.

**Events (US government / Fed):**
- **FOMC** meeting decision dates — Federal Reserve
- **CPI** release dates — BLS (via FRED release id 10)
- **PPI** release dates — BLS (via FRED release id 46)
- **Non-Farm Payroll** = *Employment Situation* release dates — BLS (via FRED release id 50)

**Sources:**
- CPI / PPI / NFP: FRED `release/dates` API — `https://api.stlouisfed.org/fred/release/dates`.
  Returns the actual publication date of each release (the announcement we want).
- FOMC: scraped from the Federal Reserve site — `fomchistorical{year}.htm` (2010–2020) and
  `fomccalendars.htm` (2021+). The **second day** of each two-day meeting (the statement day)
  is used as the decision date.

**Window:** 2010-06-08 → 2026-06-01 (matches the XAU/USD price window).

**Output:** `macro_event_indicators.parquet` — a business-day (`Mon–Fri`) tz-naive `Date` index
with a single column `macro_event` = `1` if **any** of the four events occurs that day, else `0`.

> **Prerequisite:** the FRED endpoint returns HTTP 403 without an API key. Get a free key at
> https://fred.stlouisfed.org/docs/api/api_key.html and set it as the `FRED_API_KEY` environment
> variable (or paste it into the config cell below). No key is needed for the FOMC scrape.

In [1]:
# Cell 1 — Install dependencies
%pip install -q pandas requests pyarrow lxml


[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Cell 2 — Configuration
import os
import re
from datetime import datetime
import requests
import pandas as pd

START_DATE = datetime(2010, 6, 8)
END_DATE   = datetime(2026, 6, 1)

# Free FRED API key: https://fred.stlouisfed.org/docs/api/api_key.html
# Either `export FRED_API_KEY=...` in your shell, or paste the key into the fallback string below.
FRED_API_KEY = ""  # ADD API KEY IF USING THIS NOTEBOOK

# FRED release ids for the three BLS releases (NFP = Employment Situation).
FRED_RELEASE_IDS = {"CPI": 10, "PPI": 46, "NFP": 50}

# The Federal Reserve CDN rejects default user agents; use a browser-like header (same as the
# price-data notebook).
HEADERS = {"User-Agent": "Mozilla/5.0 (research data download)"}

assert FRED_API_KEY, (
    "No FRED_API_KEY found. Get a free key at "
    "https://fred.stlouisfed.org/docs/api/api_key.html and set the FRED_API_KEY environment "
    "variable (or paste it into FRED_API_KEY above)."
)
print("Window:", START_DATE.date(), "->", END_DATE.date())
print("FRED key loaded:", bool(FRED_API_KEY))

Window: 2010-06-08 -> 2026-06-01
FRED key loaded: True


In [3]:
# Cell 3 — FRED release dates for CPI / PPI / NFP
FRED_URL = "https://api.stlouisfed.org/fred/release/dates"

fred_event_dates = {}
for name, release_id in FRED_RELEASE_IDS.items():
    params = {
        "release_id": release_id,
        "api_key": FRED_API_KEY,
        "file_type": "json",
        "realtime_start": START_DATE.strftime("%Y-%m-%d"),
        "realtime_end": END_DATE.strftime("%Y-%m-%d"),
        "sort_order": "asc",
        "limit": 10000,
    }
    resp = requests.get(FRED_URL, params=params, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    records = resp.json().get("release_dates", [])
    dates = pd.to_datetime([r["date"] for r in records]).normalize()
    # Keep only dates inside the window (realtime filtering is on the realtime period, not the
    # release date itself).
    dates = dates[(dates >= START_DATE) & (dates <= END_DATE)]
    fred_event_dates[name] = pd.DatetimeIndex(sorted(set(dates)))
    d = fred_event_dates[name]
    print(f"{name:>4}: {len(d):>4} release dates   {d.min().date()} -> {d.max().date()}")

 CPI:  203 release dates   2010-06-17 -> 2026-05-12
 PPI:  201 release dates   2010-06-16 -> 2026-05-13
 NFP:  194 release dates   2010-07-02 -> 2026-05-08


In [4]:
# Cell 4 — FOMC meeting decision dates (scrape the Federal Reserve site)
#
# Pages: fomchistorical{year}.htm for 2010-2020, fomccalendars.htm for 2021+.
# Meetings render as a month + a two-day range, e.g. "January 27-28" or the month-spanning
# "January/February 31-1". We take the SECOND day of each range (the statement/decision day).

MONTHS = {
    "january": 1, "february": 2, "march": 3, "april": 4, "may": 5, "june": 6,
    "july": 7, "august": 8, "september": 9, "october": 10, "november": 11, "december": 12,
}
# Matches e.g. "January 27-28", "January/February 31-1", "Jan. 27-28" with optional trailing '*'.
MEETING_RE = re.compile(
    r"(January|February|March|April|May|June|July|August|September|October|November|December)"
    r"(?:/(January|February|March|April|May|June|July|August|September|October|November|December))?"
    r"\s+(\d{1,2})-(\d{1,2})",
    re.IGNORECASE,
)


def parse_fomc_year(html, year):
    """Return the list of decision dates (2nd day of each meeting) found for `year` in `html`."""
    found = []
    for m in MEETING_RE.finditer(html):
        first_month = MONTHS[m.group(1).lower()]
        second_month = MONTHS[m.group(2).lower()] if m.group(2) else first_month
        second_day = int(m.group(4))
        # The decision date is the second day; its month is the second month if the range
        # spans a month boundary. A Dec/Jan meeting's 2nd day belongs to the NEXT year.
        dec_year = year + 1 if (first_month == 12 and second_month == 1) else year
        try:
            found.append(pd.Timestamp(dec_year, second_month, second_day))
        except ValueError:
            continue  # skip impossible dates from any layout noise
    return found


fomc_dates = set()
years = range(START_DATE.year, END_DATE.year + 1)

# 2021+ all live on the single calendars page; fetch it once.
calendars_html = requests.get(
    "https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm",
    headers=HEADERS, timeout=30,
).text

per_year_counts = {}
for year in years:
    if year >= 2021:
        # The calendars page is organised in per-year sections; restrict the regex scan to the
        # slice for this year so months are attributed correctly.
        marker = str(year)
        start = calendars_html.find(marker)
        nxt = calendars_html.find(str(year + 1))
        chunk = calendars_html[start:nxt] if start != -1 and nxt != -1 else calendars_html
        dates = parse_fomc_year(chunk, year)
    else:
        url = f"https://www.federalreserve.gov/monetarypolicy/fomchistorical{year}.htm"
        html = requests.get(url, headers=HEADERS, timeout=30).text
        dates = parse_fomc_year(html, year)
    in_year = [d for d in dates if d.year == year]
    per_year_counts[year] = len(set(in_year))
    fomc_dates.update(dates)

# Keep only decision dates inside the window.
fomc_dates = pd.DatetimeIndex(sorted(d for d in fomc_dates if START_DATE <= d <= END_DATE))

print("FOMC decision dates parsed per year (expect ~8 scheduled):")
for year, n in per_year_counts.items():
    flag = "" if 6 <= n <= 12 else "   <-- CHECK"
    print(f"  {year}: {n}{flag}")
print(f"\nTotal FOMC decision dates in window: {len(fomc_dates)}")
print("Range:", fomc_dates.min().date(), "->", fomc_dates.max().date())
display(fomc_dates.to_frame(index=False, name="fomc_decision_date"))

FOMC decision dates parsed per year (expect ~8 scheduled):
  2010: 4   <-- CHECK
  2011: 5   <-- CHECK
  2012: 6
  2013: 8
  2014: 8
  2015: 8
  2016: 8
  2017: 6
  2018: 7
  2019: 8
  2020: 8
  2021: 0   <-- CHECK
  2022: 0   <-- CHECK
  2023: 0   <-- CHECK
  2024: 0   <-- CHECK
  2025: 0   <-- CHECK
  2026: 0   <-- CHECK

Total FOMC decision dates in window: 74
Range: 2010-06-23 -> 2020-12-16


,fomc_decision_date
0,2010-06-23
1,2010-11-03
2,2011-01-26
3,2011-04-27
4,2011-06-22
...,...
69,2020-06-10
70,2020-07-29
71,2020-09-16
72,2020-11-05


In [5]:
# Cell 5 — Build the business-day index and the single combined indicator
idx = pd.bdate_range(START_DATE, END_DATE)   # Mon-Fri, tz-naive
idx.name = "Date"

# Union of every event date across the four sources.
all_events = set()
for d in fred_event_dates.values():
    all_events.update(d)
all_events.update(fomc_dates)

# Remap any event that falls off the business-day calendar (e.g. the Sun 2020-03-15 emergency
# FOMC cut) onto the next business day so the signal is not lost.
bday_set = set(idx)
mapped_events = set()
remaps = []
for d in sorted(all_events):
    if d < START_DATE or d > END_DATE:
        continue
    if d in bday_set:
        mapped_events.add(d)
    else:
        nxt = (d + pd.offsets.BDay(1)).normalize()
        if nxt in bday_set:
            mapped_events.add(nxt)
            remaps.append((d.date(), nxt.date()))

df = pd.DataFrame(index=idx)
df["macro_event"] = df.index.isin(mapped_events).astype(int)

if remaps:
    print("Off-calendar events remapped to next business day:")
    for orig, nxt in remaps:
        print(f"  {orig} -> {nxt}")
print("\nTotal business days:", len(df))
print("Event days (macro_event == 1):", int(df['macro_event'].sum()))
print(df["macro_event"].value_counts())
display(df.head())
display(df[df["macro_event"] == 1].head())


Total business days: 4170
Event days (macro_event == 1): 657
macro_event
0    3513
1     657
Name: count, dtype: int64


,macro_event
Date,
2010-06-08,0
2010-06-09,0
2010-06-10,0
2010-06-11,0
2010-06-14,0


,macro_event
Date,
2010-06-16,1
2010-06-17,1
2010-06-23,1
2010-07-02,1
2010-07-15,1


In [6]:
# Cell 6 — Save to Parquet (+ round-trip check)
OUT_PATH = "macro_event_indicators.parquet"

df.to_parquet(OUT_PATH, engine="pyarrow")
print(f"Saved {len(df):,} rows to {OUT_PATH}")

check = pd.read_parquet(OUT_PATH)
print("Read-back shape:", check.shape)
assert check.shape == df.shape, "Round-trip shape mismatch!"
assert check["macro_event"].sum() == df["macro_event"].sum(), "Round-trip indicator mismatch!"
check.tail()

Saved 4,170 rows to macro_event_indicators.parquet
Read-back shape: (4170, 1)


,macro_event
Date,
2026-05-26,0
2026-05-27,0
2026-05-28,0
2026-05-29,0
2026-06-01,0


In [8]:
# Cell 7 — (optional) spot-check alignment with the HAR working dataset
rv = pd.read_parquet("merged_RV_GVZ.parquet")
joined = rv.join(df, how="inner")
print("merged_RV_GVZ rows:", len(rv), "| inner-join with indicators:", len(joined))
print("Event days within the joined (trading-day) sample:", int(joined['macro_event'].sum()))
display(joined.head())

OUTPATH = "merged_RV_GVZ_with_macro_event.parquet"
joined.to_parquet(OUTPATH, engine="pyarrow")



merged_RV_GVZ rows: 4114 | inner-join with indicators: 4015
Event days within the joined (trading-day) sample: 649


,RV_gold,RV_crude,RV_ES,GVZ_close,macro_event
Date,,,,,
2010-06-08,19.751379,39.009638,30.927167,24.98,0
2010-06-09,17.166936,32.664646,25.269197,24.58,0
2010-06-10,16.844851,34.403650,24.717876,23.29,0
2010-06-11,12.370461,34.640662,20.392408,21.62,0
2010-06-14,14.860902,32.545123,18.610217,21.79,0
